# NHANES Linked Mortality Data — Parsing & Consolidation

**Purpose.** Convert the NCHS public-use linked mortality files from fixed-width ASCII into a
single consolidated CSV, keyed by `SEQN`, for joining to the NHANES analytical cohort. This is
the first step of the survival-analysis chain.

**Input.** One `*_MORT_2019_PUBLIC.dat` file per NHANES cycle, downloaded from the NCHS data
linkage FTP site, in `nhanes_mortality/mortality_data/`.

**Output.** `combined_data_raw.csv` — all cycles stacked, one row per participant.

**How to run.** Set `data_directory` and `data_directory1` to your local paths, create the
`mortality_data_csv` folder, then run top to bottom. Requires `pandas` only (the `sklearn` and
`seaborn` imports are vestigial). See the execution-order caveat in Known issues before
trusting the intermediate outputs of Sections 4–5.

---

## File layout

The NCHS files are fixed-width ASCII with no header row, so the byte offsets **are** the
schema. A wrong offset produces plausible-looking but incorrect values rather than an error,
which is why the frequency tables printed for every variable matter: each has a small set of
valid codes, so a misalignment shows up immediately as nonsense.

The NHANES offsets used here were checked against the NCHS 2019 public-use layout and are
correct (positions given 1-indexed inclusive, as NCHS documents them):

| Variable | Columns | Meaning |
|---|---|---|
| `SEQN` | 1–6 | Respondent sequence number — the join key to NHANES |
| `ELIGSTAT` | 15 | 1 = eligible, 2 = under 18 (not released), 3 = ineligible |
| `MORTSTAT` | 16 | 0 = assumed alive, 1 = assumed deceased, blank = ineligible |
| `UCOD_LEADING` | 17–19 | Underlying cause recode (see below) |
| `DIABETES` | 20 | Diabetes listed among multiple causes of death (0/1) |
| `HYPERTEN` | 21 | Hypertension listed among multiple causes (0/1) |
| `PERMTH_INT` | 43–45 | Person-months of follow-up from interview date |
| `PERMTH_EXM` | 46–48 | Person-months of follow-up from MEC exam date |

`UCOD_LEADING` codes: 1 heart disease · 2 malignant neoplasms · 3 chronic lower respiratory ·
4 accidents · 5 cerebrovascular · 6 Alzheimer's · 7 diabetes mellitus · 8 influenza/pneumonia ·
9 nephritis · 10 all other causes.

---

## Known issues to review before use

Spotted while documenting and **left unchanged**, since the analysis logic was preserved as-is.
The first two are properties of the NCHS data rather than of this code, but they bear directly
on the cause-specific mortality results:

1. **Cause of death is heavily restricted for the 2015–2018 NHANES cycles.** The NCHS codebook
   states that for 2015–2018 NHANES only `UCOD_LEADING` = 001, 002 and 010 are released — that
   is, **heart disease, cancer, and "all other causes"**. Diabetes (007) and cerebrovascular
   disease (005) are *not* available for those cycles; those deaths are collapsed into 010.
   Since the cohort spans 1999–2018, this means diabetes-specific deaths are structurally
   undercounted in the two most recent cycles, and CVD defined as {1, 5} loses its
   cerebrovascular component there as well. Any cause-specific rate computed across the full
   span is therefore attenuated, and unevenly so by cycle. Worth either restricting
   cause-specific analyses to 1999–2014 or stating the limitation explicitly.
6. **`combine_csv_files` reads every CSV in the folder** with no filename filter. Because the
   `.dat` glob in Section 3 matches NHIS files as well as NHANES ones, NHIS outputs written to
   the same folder would be concatenated in — and their column set differs entirely
   (`publicid`, `dodqtr`, `dodyear`, `wgt_new`), producing large blocks of NaN.
7. **This notebook does not produce `diabetes_with_mortality.csv`.** The clinical
   characterisation notebook reads that file, but the join between `combined_data_raw.csv` and
   the clustered cohort on `SEQN` happens nowhere in this notebook. Either that step lives in an
   unshared script or it is missing from the repository — worth adding here so the chain is
   reproducible end to end.


> Cell outputs have been cleared so the notebook is light and diffs cleanly. Re-run top to
> bottom to regenerate them.


In [ ]:
!pip install scikit-learn
!pip install seaborn

In [ ]:
import pandas as pd
import warnings
from sklearn.exceptions import ConvergenceWarning
import seaborn as sns


# Targeted suppression rather than a blanket filter: only FutureWarning and
# ConvergenceWarning are hidden, so genuine errors and RuntimeWarnings still
# surface. (Note: sklearn and seaborn are imported but not actually used
# anywhere in this notebook — see Known issues.)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
# Import necessary libraries
import os
import pandas as pd

# Directory where .DAT files are saved
# NOTE: absolute Windows paths — change both for another machine.
data_directory = r"C:\Users\ridsa\Downloads\Final_Project\nhanes_mortality\mortality_data"
data_directory1 =  r"C:\Users\ridsa\Downloads\Final_Project\nhanes_mortality\mortality_data_csv"

# List all files in the directory
# CAUTION: this pattern matches BOTH NHIS_* and NHANES_* mortality files if
# both are present in the folder. Only the NHANES ones belong in this project.
files = [f for f in os.listdir(data_directory) if f.endswith('_MORT_2019_PUBLIC.dat')]

# Define a function to process each file
def process_file(file_path, survey_name):
    # Read the fixed-width format ASCII file
    # The NCHS public-use linked mortality files ship as fixed-width ASCII with
    # no header row, so the byte offsets below ARE the schema — a wrong offset
    # silently yields plausible but incorrect numbers rather than an error.
    # Offsets are 0-indexed half-open, matching the NCHS 1-indexed inclusive
    # layout: e.g. SEQN at columns 1-6 becomes (0, 6).
    # NHIS Version
    if 'NHIS' in survey_name:
        colspecs = [(0, 14), (14, 15), (15, 16), (16, 19), (19, 20), (20, 21), (21, 22), (22, 26), (26, 34), (34, 42)]
        column_names = ['publicid', 'eligstat', 'mortstat', 'ucod_leading', 'diabetes', 'hyperten', 'dodqtr', 'dodyear', 'wgt_new', 'sa_wgt_new']
    # NHANES Version
    # Verified against the NCHS 2019 public-use layout:
    #   SEQN 1-6 | ELIGSTAT 15 | MORTSTAT 16 | UCOD_LEADING 17-19 |
    #   DIABETES 20 | HYPERTEN 21 | PERMTH_INT 43-45 | PERMTH_EXM 46-48
    # Columns 7-14 and 22-42 are filler and are deliberately not read.
    elif 'NHANES' in survey_name:
        colspecs = [(0, 6), (14, 15), (15, 16), (16, 19),(19, 20), (20, 21), (42, 45), (45, 48)]
        column_names = ['SEQN', 'eligstat', 'mortstat', 'ucod_leading', 'diabetes', 'hyperten', 'permth_int', 'permth_exm']
    else:
        # Returns None (NOT a tuple), which the caller unpacks into two names —
        # see Known issues.
        print(f"Unknown survey type in file: {survey_name}")
        return None

    # na_values covers both conventions NCHS uses for missing: a blank field
    # and a literal period.
    df = pd.read_fwf(file_path, colspecs=colspecs, names=column_names, na_values=["", "."])

    # Print the structure and contents of the data
    # (df.info() prints directly and returns None, so this also prints "None".)
    print(f"Structure of {survey_name}:")
    print(df.info())
    
    # Variable Frequencies
    # Frequency tables are the practical check that the byte offsets are right:
    # every variable below has a small, known set of valid codes, so an
    # off-by-one offset shows up immediately as nonsense values.
    frequencies = {}
    
    # ELIGSTAT: Eligibility Status for Mortality Follow-up
    # 1 = Eligible, 2 = Under 18 (not released), 3 = Ineligible.
    # dropna=False so missing is counted rather than silently omitted.
    frequencies['eligstat'] = df['eligstat'].value_counts(dropna=False).to_dict()
    
    # MORTSTAT: Final Mortality Status
    # 0 = Assumed alive, 1 = Assumed deceased, missing = ineligible/under 18.
    frequencies['mortstat'] = df['mortstat'].value_counts(dropna=False).to_dict()
    
    # UCOD_LEADING: Underlying Cause of Death: Recode
    # 1 = heart disease, 2 = malignant neoplasms, 3 = chronic lower respiratory,
    # 4 = accidents, 5 = cerebrovascular, 6 = Alzheimer's, 7 = diabetes mellitus,
    # 8 = influenza/pneumonia, 9 = nephritis, 10 = all other causes.
    # IMPORTANT: codes 3-9 are NOT released for the 2015-2018 NHANES cycles —
    # those deaths appear as 10. See Known issues in the header.
    frequencies['ucod_leading'] = df['ucod_leading'].value_counts(dropna=False).to_dict()

    # DIABETES: DIABETES as a contributing cause of death
    # This is the multiple-cause-of-death FLAG (0/1), a different and broader
    # measure than ucod_leading == 7, which requires diabetes to be the single
    # underlying cause.
    frequencies['diabetes'] = df['diabetes'].value_counts(dropna=False).to_dict()

    # HYPERTEN: Hyoertension as a contributing cause of death
    frequencies['hyperten'] = df['hyperten'].value_counts(dropna=False).to_dict()

    
    # Additional NHIS-specific variables
    # NHIS records date of death as quarter + year instead of person-months,
    # so these two columns exist only for that survey.
    if 'NHIS' in survey_name:
        frequencies['dodqtr'] = df['dodqtr'].value_counts(dropna=False).to_dict()
        frequencies['dodyear'] = df['dodyear'].value_counts(dropna=False).to_dict()
    
    # Print frequency tables
    print(f"Frequencies for {survey_name}:")
    for var, freq in frequencies.items():
        print(f"{var}: {freq}")
    
    # Rename the dataframe to the shorthand name
    return df, frequencies

# Iterate over all files and process them
# One CSV per survey cycle is written to data_directory1; the next cell
# concatenates them.
for file in files:
    survey_name = os.path.splitext(file)[0]
    file_path = os.path.join(data_directory, file)
    # NOTE: this unpack raises TypeError if process_file returned None.
    df, freq = process_file(file_path, survey_name)
    # Save the dataframe to a new CSV file for convenience
    # (converting once means later cells never re-parse the fixed-width files)
    if df is not None:
        output_csv = os.path.join(data_directory1, f"{survey_name}.csv")
        df.to_csv(output_csv, index=False)
        print(f"Processed and saved {survey_name} to {output_csv}")

In [ ]:
import os
import pandas as pd

def combine_csv_files(directory):
    """Combines all CSV files in the specified directory into a single DataFrame.

    Args:
        directory (str): The path to the directory containing the CSV files.

    Returns:
        pandas.DataFrame: A DataFrame containing the combined data from all CSV files.
                             Returns None if no CSV files are found.
    """
    # Stacks every cycle into one long frame keyed by SEQN, which is unique
    # across all continuous-NHANES cycles, so no cycle identifier is needed to
    # keep participants distinct.
    # CAUTION: this reads EVERY .csv in the folder with no filename filter. If
    # NHIS outputs were written here too, their different column set would be
    # union-ed in and produce large blocks of NaN.

    all_data = []
    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath)
            all_data.append(df)

    if all_data:
        # ignore_index=True renumbers rows 0..N-1; the original per-file index
        # carries no meaning once the cycles are stacked.
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df
    else:
        print("No CSV files found in the directory.")
        return None

# --- Example Usage ---

data_directory = r"C:\Users\ridsa\Downloads\Final_Project\nhanes_mortality\mortality_data_csv" 

combined_data = combine_csv_files(data_directory)

# Written back to the .dat folder (not the _csv folder) as the single
# consolidated file that the rest of the pipeline reads.
if combined_data is not None:
    output_path = os.path.join(r"C:\Users\ridsa\Downloads\Final_Project\nhanes_mortality\mortality_data", "combined_data_raw.csv")
    combined_data.to_csv(output_path, index=False)
    print(f"Combined data saved to: {output_path}")

In [ ]:
# Exclude participants where 'MORTSTAT' is missing since they were not eligible for linkage, so we don’t know their status.
# Missing mortstat means ineligible or under 18, i.e. no vital-status
# information at all — such records cannot contribute person-time and would
# bias any rate downward if retained.
# CAUTION: `df` here is still bound to the LAST file processed by the loop in
# the cell above, not to the combined dataset — the combined file is only
# loaded into `df` two cells further down. See Known issues.

df_mort = df[df['mortstat'].notna()]
df_mort

In [ ]:
# Subset of confirmed deaths (mortstat == 1, "assumed deceased").
# Inspected as a sanity check on the event count before any rate is computed.
deceased = df_mort[df_mort['mortstat'] == 1]
deceased

In [ ]:
# Load the consolidated mortality file written above.
# NOTE: relative path, unlike the absolute paths used in the two cells above —
# this only resolves if the working directory is the project root.
# This is also the point at which `df` finally refers to ALL cycles.
df = pd.read_csv('nhanes_mortality/mortality_data/combined_data_raw.csv')

In [ ]:
# Display the combined frame — row count should equal the sum of the
# per-cycle files written earlier.
df

In [ ]:
import pandas as pd

# Assuming df_dropped is your DataFrame
# Replace NaN in 'ucod_leading' with 0 where 'mortstat' is 0
# Rationale: survivors have no cause of death, so NCHS leaves ucod_leading
# blank for them. Filling with 0 distinguishes "alive, no cause applicable"
# from "died, cause unknown" (which stays NaN) — without this, a later
# `.dropna()` on cause of death would silently discard every survivor.
# NOTE: 0 is NOT a value in the NCHS codebook (valid codes are 001-010), so it
# is a local convention; make sure downstream code never treats 0 as a cause.
df.loc[(df['ucod_leading'].isna()) & (df['mortstat'] == 0), 'ucod_leading'] = 0

# Display the result_
print(df)

In [ ]:
# Final check on the outcome variable: expect {0, 1} plus NaN for the
# ineligible records that were not filtered out of this frame.
df['mortstat'].unique()